<a href="https://colab.research.google.com/github/kemal1308/TugasBesarBigData-hm/blob/main/notebooks/03_model_tuning_and_evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Menghubungkan Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Inisialisasi Spark Session dengan alokasi memori maksimal untuk Colab
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder \
    .appName("HM_Hyperparameter_Tuning") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "10g") \
    .config("spark.sql.shuffle.partitions", "100") \
    .getOrCreate()

# 3. Membaca data Parquet hasil Data Engineer (Anggota 1)
# Sesuaikan path jika Anda menggunakan shortcut langsung
parquet_path = "/content/drive/MyDrive/Dataset_HM/cleaned_data.parquet"
df_raw = spark.read.parquet(parquet_path)

print(f"Data awal berhasil dimuat: {df_raw.count()} baris.")

Mounted at /content/drive
Data awal berhasil dimuat: 31788324 baris.


In [2]:
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline

# 1. Filtering waktu yang sama dengan Anggota 2 (menyusutkan ke ~2 juta baris aktif)
df_active = df_raw.filter(F.col("t_dat") >= "2020-08-01")

# 2. Membuat metrik Implicit Feedback (frekuensi pembelian)
interactions = df_active.groupBy("customer_id", "article_id") \
    .agg(F.count("*").alias("purchase_count"))

# 3. Mengubah ID Teks menjadi Indeks Angka (StringIndexer)
user_indexer = StringIndexer(inputCol="customer_id", outputCol="user_idx", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="article_id", outputCol="item_idx", handleInvalid="skip")

pipeline = Pipeline(stages=[user_indexer, item_indexer])
indexed_df = pipeline.fit(interactions).transform(interactions)

# 4. Membagi data (80% Train, 20% Test) dengan seed yang sama (42)
train_df, test_df = indexed_df.randomSplit([0.8, 0.2], seed=42)

# Mengunci data di memori agar proses iterasi (tuning) lebih cepat
train_df.cache()
test_df.cache()

print(f"Data siap di-tuning! Train: {train_df.count()} baris | Test: {test_df.count()} baris")

Data siap di-tuning! Train: 1434653 baris | Test: 358961 baris


In [3]:
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit

# 1. Mengambil sampel 20% dari data latih
tune_df = train_df.sample(fraction=0.2, seed=42)
tune_df.cache()

# 2. Inisialisasi model dasar
als = ALS(userCol="user_idx", itemCol="item_idx", ratingCol="purchase_count",
          implicitPrefs=True, alpha=40.0, coldStartStrategy="drop")

# 3. Menyusun Grid 4 Kombinasi)
param_grid = ParamGridBuilder() \
    .addGrid(als.rank, [10, 20]) \
    .addGrid(als.maxIter, [10]) \
    .addGrid(als.regParam, [0.05, 0.1]) \
    .build()

evaluator = RegressionEvaluator(metricName="rmse", labelCol="purchase_count", predictionCol="prediction")

tvs = TrainValidationSplit(estimator=als,
                           estimatorParamMaps=param_grid,
                           evaluator=evaluator,
                           trainRatio=0.8)

print("Memulai Hyperparameter Tuning Cepat (4 Kombinasi pada subset 20% data)...")
tvs_model = tvs.fit(tune_df)
print("Proses Tuning Selesai!")

Memulai Hyperparameter Tuning Cepat (4 Kombinasi pada subset 20% data)...
Proses Tuning Selesai!


In [4]:
# 1. Mengambil kombinasi parameter terbaik
best_rank = tvs_model.bestModel._java_obj.parent().getRank()
best_iter = tvs_model.bestModel._java_obj.parent().getMaxIter()
best_reg = tvs_model.bestModel._java_obj.parent().getRegParam()

print("==================================================")
print("HYPERPARAMETER TERBAIK DITEMUKAN:")
print(f"- Rank     : {best_rank}")
print(f"- MaxIter  : {best_iter}")
print(f"- RegParam : {best_reg}")
print("==================================================")

# 2. Latih ulang model menggunakan parameter terbaik (1,4 juta baris)
print("Melatih model final dengan parameter terbaik pada seluruh data latih...")
final_als = ALS(userCol="user_idx", itemCol="item_idx", ratingCol="purchase_count",
                implicitPrefs=True, alpha=40.0, coldStartStrategy="drop",
                rank=best_rank, maxIter=best_iter, regParam=best_reg)

best_als_model_full = final_als.fit(train_df)

# 3. Evaluasi pada data Test (20% dari total data asli)
predictions = best_als_model_full.transform(test_df)
final_rmse = evaluator.evaluate(predictions)

print(f"Skor Baseline sebelumnya : 1.0794")
print(f"Skor RMSE Final (Tuned)  : {final_rmse:.4f}")

# 4. Simpan Model (Opsional)
MODEL_DIR = "/content/drive/MyDrive/Dataset_HM/models/best_als_model"
best_als_model_full.write().overwrite().save(MODEL_DIR)
print(f"Model final disimpan di: {MODEL_DIR}")

HYPERPARAMETER TERBAIK DITEMUKAN:
- Rank     : 20
- MaxIter  : 10
- RegParam : 0.05
Melatih model final dengan parameter terbaik pada seluruh data latih...
Skor Baseline sebelumnya : 1.0794
Skor RMSE Final (Tuned)  : 1.0807
Model final disimpan di: /content/drive/MyDrive/Dataset_HM/models/best_als_model
